In [10]:
# ==================== country_standardizer.py ====================
import re
import unicodedata
import pandas as pd
import pycountry
import phonenumbers

In [11]:
# ---------- Normalisatie: case/accents/spaties/streepjes/puntjes eruit ----------
def _norm(s: str) -> str:
    if not isinstance(s, str):
        return ""
    s = s.strip().casefold()
    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    # verwijder alles behalve letters/cijfers
    s = re.sub(r"[^a-z0-9]", "", s)
    return s

In [12]:
print(_norm("Antigua-Barbuda"))      # → 'antiguabarbuda'
print(_norm("Türkiye"))              # → 'turkiye'
print(_norm("Côte d’Ivoire"))        # → 'cotedivoire'
print(_norm(" St. Vincent "))        # → 'stvincent'
print(_norm("U.S.A."))               # → 'usa'

#toepassen op datafram flag_raw["name_norm"] = flag_raw["name"].apply(_norm)


antiguabarbuda
turkiye
cotedivoire
stvincent
usa


In [13]:
def get_phone_code(alpha2: str) -> str | None:
    try:
        code = phonenumbers.country_code_for_region(alpha2)
        return f"+{code}" if code else None
    except Exception:
        return None

In [14]:
print(get_phone_code("NL"))  # → +31
print(get_phone_code("TR"))  # → +90
print(get_phone_code("US"))  # → +1
print(get_phone_code("ZZ"))  # → None (bestaat niet)

+31
+90
+1
None


In [ ]:
# ---------- Alias-lijst: historische/alternatieve namen → ISO2 ----------
# Definieer elke alias maar 1x in normale leesbare vorm; normalisatie doet de rest.
ALIASES_HUMAN = {
    # === jouw concrete lijst ===
    "british virgin isles": "VG",
    "cape verde islands": "CV",
    "comorro islands": "KM",
    "faeroes": "FO",
    "falklands malvinas": "FK",
    "germany ddr": "DE",
    "germany frg": "DE",
    "kampuchea": "KH",
    "malagasy": "MG",
    "maldive islands": "MV",
    "marianas": "MP",                    # Northern Mariana Islands
    "netherlands antilles": "CW",        # pragmatisch → Curaçao
    "north yemen": "YE",
    "parguay": "PY",                     # typfout → Paraguay
    "soloman islands": "SB",             # typfout → Solomon Islands
    "south yemen": "YE",
    "st helena": "SH",
    "st kitts nevis": "KN",
    "st lucia": "LC",
    "st vincent": "VC",
    "trinidad tobago": "TT",
    "turkey": "TR",                      # → Türkiye
    "turks cocos islands": "TC",         # Turks and Caicos Islands
    "uae": "AE",
    "us virgin isles": "VI",

    # === veelvoorkomende historische/rename varianten ===
    "ussr": "RU",
    "soviet union": "RU",
    "zaire": "CD",
    "yugoslavia": "RS",                  # pragmatisch (alternatief: opsplitsen)
    "czechoslovakia": "CZ",              # pragmatisch (alternatief: SK)
    "burma": "MM",                       # → Myanmar
    "western samoa": "WS",               # → Samoa
    "swaziland": "SZ",                   # → Eswatini
    "cape verde": "CV",                  # → Cabo Verde
    "macedonia": "MK",                   # → North Macedonia
    "fyrom": "MK",
    "east timor": "TL",                  # → Timor-Leste
    "ivory coast": "CI",                 # → Côte d’Ivoire
    "lao pdr": "LA",
    "moldavia": "MD",                    # → Moldova
    "byelorussia": "BY",                 # → Belarus
    "malaya": "MY",                      # → Malaysia
    "ceylon": "LK",                      # → Sri Lanka
    "siam": "TH",                        # → Thailand
    "rhodesia": "ZW",                    # → Zimbabwe
    "british honduras": "BZ",            # → Belize
    "upper volta": "BF",                 # → Burkina Faso
    "dahomey": "BJ",                     # → Benin
    "persia": "IR",                      # historisch → Iran

    # === ambigue/varianten (Congo, UK/US, Korea, etc.) ===
    "congo kinshasa": "CD",
    "democratic republic of congo": "CD",
    "drc": "CD",
    "congo brazzaville": "CG",
    "republic of congo": "CG",
    "the congo": "CG",
    "uk": "GB",
    "u k": "GB",
    "great britain": "GB",
    "united states of america": "US",
    "usa": "US",
    "u s a": "US",
    "south korea": "KR",
    "republic of korea": "KR",
    "north korea": "KP",
    "democratic people s republic of korea": "KP",

    # === Saint-varianten (zonder punt, &-varianten) ===
    "saint helena": "SH",
    "saint kitts and nevis": "KN",
    "saint lucia": "LC",
    "saint vincent and the grenadines": "VC",
    "trinidad and tobago": "TT",
    "antigua and barbuda": "AG",
    "st kitts and nevis": "KN",
    "st vincent and the grenadines": "VC",
    "st maarten": "SX",
    "st. maarten": "SX",
    "saint martin (dutch part)": "SX",
    "saint martin (french part)": "MF",

    # === overig handig ===
    "holland": "NL",
    "curacao": "CW",
    "curaçao": "CW",
    "greenland": "GL",
    "faroe islands": "FO",
    "falkland islands": "FK",
    "antigua barbuda": "AG",
    "antigua-barbuda": "AG",    
    "Antigua-Barbuda": "AG"
}


In [17]:
# Normale alias-map → genormaliseerde sleutel → ISO2
ALIAS_TO_ISO2 = {_norm(k): v for k, v in ALIASES_HUMAN.items()}
print(ALIAS_TO_ISO2)

{'britishvirginisles': 'VG', 'capeverdeislands': 'CV', 'comorroislands': 'KM', 'faeroes': 'FO', 'falklandsmalvinas': 'FK', 'germanyddr': 'DE', 'germanyfrg': 'DE', 'kampuchea': 'KH', 'malagasy': 'MG', 'maldiveislands': 'MV', 'marianas': 'MP', 'netherlandsantilles': 'CW', 'northyemen': 'YE', 'parguay': 'PY', 'solomanislands': 'SB', 'southyemen': 'YE', 'sthelena': 'SH', 'stkittsnevis': 'KN', 'stlucia': 'LC', 'stvincent': 'VC', 'trinidadtobago': 'TT', 'turkey': 'TR', 'turkscocosislands': 'TC', 'uae': 'AE', 'usvirginisles': 'VI', 'ussr': 'RU', 'sovietunion': 'RU', 'zaire': 'CD', 'yugoslavia': 'RS', 'czechoslovakia': 'CZ', 'burma': 'MM', 'westernsamoa': 'WS', 'swaziland': 'SZ', 'capeverde': 'CV', 'macedonia': 'MK', 'fyrom': 'MK', 'easttimor': 'TL', 'ivorycoast': 'CI', 'laopdr': 'LA', 'moldavia': 'MD', 'byelorussia': 'BY', 'malaya': 'MY', 'ceylon': 'LK', 'siam': 'TH', 'rhodesia': 'ZW', 'britishhonduras': 'BZ', 'uppervolta': 'BF', 'dahomey': 'BJ', 'persia': 'IR', 'congokinshasa': 'CD', 'democr

In [ ]:
# ---------- pycountry index (officiële / common / official namen) ----------
#c.name           # officiële landnaam (bv. "Netherlands")
#c.alpha_2        # ISO2-code (bv. "NL")
#c.alpha_3        # ISO3-code (bv. "NLD")
#c.numeric        # numerieke code (bv. "528")
#c.official_name  # officiële lange naam (bv. "Kingdom of the Netherlands") [soms aanwezig]
#c.common_name    # alternatieve korte naam (bv. "Bolivia") [soms aanwezig]

_NAME_TO_ISO2: dict[str, str] = {}
for c in pycountry.countries:
    _NAME_TO_ISO2[_norm(c.name)] = c.alpha_2
    for attr in ("official_name", "common_name"):
        if hasattr(c, attr):
            _NAME_TO_ISO2[_norm(getattr(c, attr))] = c.alpha_2

In [21]:
def standardize_country(value: str) -> tuple[str | None, str | None, str | None]:
    """
    Herkent land op basis van:
      1) alias-lijst (historisch/alternatief),
      2) ISO2/ISO3,
      3) officiële/common/official naam (pycountry),
    en retourneert (country_name, ISO2, phone_code). Geen match → (None, None, None).
    """
    if pd.isna(value):
        return None, None, None

    key = _norm(str(value))

    # 1) alias-match
    iso2 = ALIAS_TO_ISO2.get(key)
    if iso2:
        c = pycountry.countries.get(alpha_2=iso2)
        return c.name, iso2, get_phone_code(iso2)

    # 2) directe ISO2 / ISO3
    for c in pycountry.countries:
        if key == _norm(c.alpha_2):
            return c.name, c.alpha_2, get_phone_code(c.alpha_2)
        if hasattr(c, "alpha_3") and key == _norm(c.alpha_3):
            return c.name, c.alpha_2, get_phone_code(c.alpha_2)

    # 3) exacte naam (officieel/common/official)
    iso2 = _NAME_TO_ISO2.get(key)
    if iso2:
        c = pycountry.countries.get(alpha_2=iso2)
        return c.name, iso2, get_phone_code(iso2)

    # 4) geen match
    return None, None, None


In [22]:
def standardize_flags(flag_raw: pd.DataFrame,
                      name_col: str = "name",
                      add_matched_flag: bool = True) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Maakt een NIEUWE DataFrame 'flags_std' met gestandaardiseerde landkolommen.
    Retourneert: (flags_std, missing_df)
    - flags_std: originele kolommen + country_name_std, country_code_std, phone_code, matched (optioneel)
    - missing_df: unieke problematische namen die niet gematcht zijn
    """
    flags_std = flag_raw.copy()
    flags_std[["country_name_std", "country_code_std", "phone_code"]] = (
        flags_std[name_col].apply(lambda x: pd.Series(standardize_country(x)))
    )

    if add_matched_flag:
        flags_std["matched"] = flags_std["country_code_std"].notna()

    # lijst met unieke niet-gematchte namen
    missing_df = (
        flags_std.loc[flags_std["country_code_std"].isna(), [name_col]]
        .dropna()
        .drop_duplicates()
        .sort_values(by=name_col)
        .rename(columns={name_col: "unmatched_name"})
        .reset_index(drop=True)
    )
    return flags_std, missing_df


In [23]:
flags_raw = pd.read_csv("flags_raw.csv")
flags_std = flags_raw.copy()

flags_std[["country_name_std", "country_code_std", "phone_code"]] = (
    flags_std["name"].apply(lambda x: pd.Series(standardize_country(x)))
)
flags_std["matched"] = flags_std["country_code_std"].notna()

# === 5) Alle ‘uitzonderingen’ (nog onopgelost) tonen ========================
not_matched = flags_std.loc[~flags_std["matched"], "name"].dropna().unique()
print("Nog te mappen uitzonderingen:", not_matched)


Nog te mappen uitzonderingen: ['Brunei' 'Burkina' 'Micronesia' 'Sao-Tome' 'Surinam' 'Vatican-City']


In [25]:
flags_raw["name_norm"] = flags_raw["name"].apply(_norm)
missing_flags = flags_std[flags_std["country_code_std"].isna()]
pd.set_option("display.max_rows", None)
print(missing_flags)

             name  flag_nr_colors flag_mainhue flag_topleft_color  \
24         Brunei               4         gold              white   
26        Burkina               3          red                red   
112    Micronesia               2         blue               blue   
144      Sao-Tome               4        green              green   
162       Surinam               4          red              green   
185  Vatican-City               4         gold               gold   

    flag_botright_color country_name_std country_code_std phone_code  matched  
24                 gold             None             None       None    False  
26                green             None             None       None    False  
112                blue             None             None       None    False  
144               green             None             None       None    False  
162               green             None             None       None    False  
185               white             

In [ ]:
import re
import unicodedata

def _norm(s: str) -> str:
    """Normaliseert tekst: lowercase, accenten/spaties/streepjes/puntjes verwijderen."""
    if not isinstance(s, str):
        return ""
    s = s.strip().casefold()  # lowercase + trimmen
    s = unicodedata.normalize("NFKD", s)  # accenten splitsen
    s = "".join(ch for ch in s if not unicodedata.combining(ch))  # accenten verwijderen
    s = re.sub(r"[^a-z0-9]", "", s)  # verwijder alles behalve letters/cijfers
    return s


In [6]:
print(_norm("Antigua-Barbuda"))      # → 'antiguabarbuda'
print(_norm("Türkiye"))              # → 'turkiye'
print(_norm("Côte d’Ivoire"))        # → 'cotedivoire'
print(_norm(" St. Vincent "))        # → 'stvincent'
print(_norm("U.S.A."))               # → 'usa'


antiguabarbuda
turkiye
cotedivoire
stvincent
usa


In [9]:
import pandas as pd

# Zet dictionary om naar DataFrame
alias_df = pd.DataFrame([
    {"alias_name": k, "country_code_std": v}
    for k, v in ALIASES_HUMAN.items()
])

# Voeg lege kolommen toe (die jij later kunt invullen)
alias_df["country_name_std"] = ""
alias_df["phone_code"] = ""
alias_df["comment"] = ""

# Sorteer op alias_name (optioneel)
alias_df = alias_df.sort_values(by="alias_name").reset_index(drop=True)

# Schrijf naar CSV
alias_df.to_csv("country_aliases_manual.csv", index=False)
print("✅ CSV-lijst 'country_aliases_manual.csv' aangemaakt!")


✅ CSV-lijst 'country_aliases_manual.csv' aangemaakt!
